<a href="https://colab.research.google.com/github/Ayman163/Hhh/blob/main/yolov10x.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U ultralytics opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.8 MB/s eta 0:00:00


In [ ]:
import cv2
import time
import queue
import threading
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import torch
from ultralytics import YOLO
from google.colab import files

class Config:
    MODEL      = "yolov10x.pt"
    CONF       = 0.45
    IOU        = 0.50
    INPUT_SIZE = 1280

    CLASSES = [0, 2, 3, 7]  # 0: person, 2: car, 3: motorcycle, 7: truck
    NAMES   = {0: "person", 2: "car", 3: "motorcycle", 7: "truck"}
    COLORS  = {
        "person":     (255, 180,   0),
        "car":        (0,   220,   0),
        "motorcycle": (0,   220, 220),
        "truck":      (180,  80,   0),
    }

    LOCK_AFTER_SECONDS = 5.0
    MIN_W_RATIO = 0.025
    MIN_H_RATIO = 0.025
    MAX_AREA_R  = 0.40


def make_botsort(cfg: Config) -> str:

    p = Path("botsort.yaml")
    p.write_text(
        "tracker_type: botsort\n"
        f"track_high_thresh: {cfg.CONF}\n"
        "track_low_thresh: 0.10\n"
        f"new_track_thresh: {cfg.CONF + 0.05:.2f}\n"
        "track_buffer: 120\n"
        "match_thresh: 0.85\n"
        "fuse_score: True\n"
        "with_reid: False\n"
        "proximity_thresh: 0.5\n"
        "appearance_thresh: 0.25\n"
        "gmc_method: none\n"
    )
    return str(p)


#TimeLocker
class TimeLocker:
    def __init__(self, cfg: Config, fps: float):
        self.lock_frames = int(fps * cfg.LOCK_AFTER_SECONDS)
        self._first  : Dict[int, int]   = {}
        self._last   : Dict[int, int]   = {}
        self._counts : Dict[int, Dict[int, int]] = {}
        self._locked : Dict[int, int]   = {}

    def get(self, tid: int, cls: int, frame_idx: int) -> Tuple[int, bool]:
        self._last[tid] = frame_idx

        if tid in self._locked:
            return self._locked[tid], True

        if tid not in self._first:
            self._first[tid] = frame_idx
            self._counts[tid] = defaultdict(int)

        self._counts[tid][cls] += 1

        if frame_idx - self._first[tid] >= self.lock_frames:
            best = max(self._counts[tid], key=self._counts[tid].get)
            self._locked[tid] = best
            self._counts.pop(tid, None)
            return best, True

        best = max(self._counts[tid], key=self._counts[tid].get)
        return best, False

    def cleanup(self, frame_idx: int, max_age: int = 300):

        old = [t for t, f in self._last.items() if frame_idx - f > max_age]
        for t in old:
            self._first.pop(t, None)
            self._last.pop(t, None)
            self._counts.pop(t, None)
            self._locked.pop(t, None)


# GPU Vectorized Drawing
def process(frame, result, frame_idx, locker, cfg, fw, fh) -> List[dict]:
    data = []

    if result.boxes is None or result.boxes.id is None:
        return data

    min_w    = fw * cfg.MIN_W_RATIO
    min_h    = fh * cfg.MIN_H_RATIO
    max_area = fw * fh * cfg.MAX_AREA_R

    #take it tensors
    xyxy  = result.boxes.xyxy
    ids   = result.boxes.id
    confs = result.boxes.conf
    clss  = result.boxes.cls

    w = xyxy[:, 2] - xyxy[:, 0]
    h = xyxy[:, 3] - xyxy[:, 1]
    area = w * h

    size_mask = (w >= min_w) & (h >= min_h) & (area <= max_area)
    class_mask = torch.isin(clss, torch.tensor(cfg.CLASSES, device=clss.device))
    combined_mask = size_mask & class_mask

    xyxy_np  = xyxy[combined_mask].cpu().numpy()
    ids_np   = ids[combined_mask].cpu().numpy().astype(int)
    confs_np = confs[combined_mask].cpu().numpy()
    clss_np  = clss[combined_mask].cpu().numpy().astype(int)

    font = cv2.FONT_HERSHEY_SIMPLEX
    thick = 1

    for box, tid, conf, cls in zip(xyxy_np, ids_np, confs_np, clss_np):
        x1, y1, x2, y2 = map(int, box)
        bw, bh = x2 - x1, y2 - y1

        stable_cls, locked = locker.get(tid, cls, frame_idx)
        name  = cfg.NAMES.get(stable_cls, "unknown")
        color = cfg.COLORS.get(name, (200, 200, 200))

        #drw
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        label = f"ID:{tid} {name} {conf:.0%}"
        scale = 0.50

        est_tw = len(label) * 9
        if est_tw > bw - 4:
            scale = max(0.30, scale * (bw - 4) / max(est_tw, 1))

        (tw, th), _ = cv2.getTextSize(label, font, scale, thick)

        ty = y1 - 4 if y1 - th - 8 >= 0 else y1 + th + 6
        cv2.rectangle(frame, (x1, ty-th-3), (x1+tw+4, ty+2), color, -1)
        cv2.putText(frame, label, (x1+2, ty), font, scale, (0,0,0), thick, cv2.LINE_AA)

        data.append({
            "id"      : tid,
            "cls_id"  : stable_cls,
            "cls_name": name,
            "locked"  : bool(locked),
            "bbox"    : (x1, y1, x2, y2),
            "center"  : ((x1+x2)/2.0, (y1+y2)/2.0),
            "conf"    : float(conf),
        })

    return data


#Multithreaded Workers

def reader_worker(cap: cv2.VideoCapture, frame_queue: queue.Queue, stop_event: threading.Event):
    while not stop_event.is_set():
        ret, frame = cap.read()
        if not ret:
            frame_queue.put(None)
            break
        frame_queue.put(frame)


def writer_worker(out: cv2.VideoWriter, result_queue: queue.Queue, locker: TimeLocker, cfg: Config, W: int, H: int, track_log: dict):
    frame_idx = 0
    while True:
        item = result_queue.get()
        if item is None:
            break
        frame, result = item
        frame_idx += 1

        frame_data = process(frame, result, frame_idx, locker, cfg, W, H)
        track_log[frame_idx] = frame_data

        if frame_idx % 300 == 0:
            locker.cleanup(frame_idx)

        out.write(frame)
        result_queue.task_done()



#Main
def run() -> Dict[int, List[dict]]:
    cfg = Config()

    print("upload video")
    uploaded   = files.upload()
    if not uploaded:
        print("upload fiald")
        return {}
    video_path = next(iter(uploaded.keys()))

    print("yolo model:")
    model       = YOLO(cfg.MODEL)
    tracker_cfg = make_botsort(cfg)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Error: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or 1280
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720

    locker = TimeLocker(cfg, fps)

    out = cv2.VideoWriter(
        "tracked_output.mp4",
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps, (W, H),
    )

    print(f"procsing: {cfg.MODEL} | {W}x{H} | {fps:.0f} FPS")
    print(f"time to locker: {cfg.LOCK_AFTER_SECONDS} second ({locker.lock_frames} fream)")

    track_log: Dict[int, List[dict]] = {}

    frame_queue = queue.Queue(maxsize=4)
    result_queue = queue.Queue(maxsize=4)
    stop_event = threading.Event()

    r_thread = threading.Thread(target=reader_worker, args=(cap, frame_queue, stop_event))
    w_thread = threading.Thread(target=writer_worker, args=(out, result_queue, locker, cfg, W, H, track_log))

    r_thread.start()
    w_thread.start()

    frame_idx = 0
    t0 = time.time()

    try:
        while True:
            frame = frame_queue.get()
            if frame is None:
                break

            frame_idx += 1

            result = model.track(
                frame,
                persist = True,
                tracker = tracker_cfg,
                conf    = cfg.CONF,
                iou     = cfg.IOU,
                imgsz   = cfg.INPUT_SIZE,
                classes = cfg.CLASSES,
                verbose = False,
            )[0]

            result_queue.put((frame, result))

            if frame_idx % 100 == 0:
                e = time.time() - t0
                print(f"procsing {frame_idx:5d} frame | spead: {frame_idx/e:5.1f} FPS")

    finally:
        stop_event.set()

        while not frame_queue.empty():
            try:
                frame_queue.get_nowait()
            except queue.Empty:
                break
        r_thread.join()

        result_queue.put(None)
        w_thread.join()

        cap.release()
        out.release()

    e = time.time() - t0
    print(f"\n done")
    print(f" avrg frame: {frame_idx} frame | time : {e:.1f} second | avrg speed  : {frame_idx/e:.1f} FPS")

    print("download ")
    files.download("tracked_output.mp4")

    return track_log


if __name__ == "__main__":
    log = run()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📤 يرجى رفع ملف الفيديو المراد معالجته...


Saving 1.mp4 to 1.mp4
⚙️ جاري تحميل نموذج YOLOv10 وتكوين المتعقب...
🚀 المعالجة ستبدأ: yolov10x.pt | 1080x1920 | 30 FPS
⏱️ سيتم تثبيت الصنف بعد: 5.0 ثوانٍ (149 فريم)
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 247ms
Prepared 1 package in 46ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

📹 تم معالجة   100 فريم | السرعة الفعلية الحالية:   6.5 FPS

✅ المعالجة انتهت بالكامل!
🎬 إجمالي الفريمات: 173 فريم | الوقت المستغرق: 24.4 ثانية | متوسط السرعة الكلي: 7.1 FPS
📥 جاري تحميل الفيديو الناتج المسمى: tracked_output.mp4...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>